## تنويه:

الشرح التالي وما يليه ,هو ترجمة مأخوذة عن دفاتر جوبتر المتاحة بالرابط:
https://github.com/fchollet/deep-learning-with-python-notebooks

وهو ملخص عملي لفقرات الكتاب:
**Deep Learning with Python**


# نظرة أولى على الشبكات العصبونية ضمن Keras

في التالي هناك مثال متكامل لاستخدام الشبكة العصبونية بوساطة بايثون والمكتبة كيراس وذلك بهدف تصنيف الأرقام المكتوبة والمعطاة كصور رمادية .

ومالم يكن لديك خبرة مسبقة بالتعامل مع كيراس , فلن تفهم كل شيء حول هذا المثال مباشرة . وربما لم تنصب كيراس حتى الآن. لا تقلق، بالفصل التالي سنراجع كل شيء بهذا المثال ونشرحهم بالتفصيل . لذلك لابأس ببعض الغموض الآن ، لأنه لا مفر من البداية من مكان ما.

 المسألة التي نريد حلها هنا هي تصنيف صور رمادية للأعداد المكتوبة يدوياً بحجم 28X28 بكسل ل10 فئات ناتجة  من0 إلى 9.
 قاعدة البانات التي سنستخدمها هي قاعدة بيانات   MNIST ، قاعدة بيانات كلاسيكية بمجتمع تعليم الآلة ، والتي كانت موجودة منذ بدء هذا المجال تقريباً ولقد تم دراستها باستفاضة . وهي مجموعة من 60,000 صورة تدريب، بالإضافة ل 10,000 صورة اختبار، تم جمعها من قبل المعهد الوطني للمعايير والتكنولوجيا NIST في MNIST في ال 1980. يمكنك اعتبار حل مسألة MNIST كبرنامج Hello World للتعلم العميق -- وهذا ما تفعله للتأكيد بأن خوارزميتك تعمل كالمطلوب . وبما أنك ستصبح ممارساً لتعليم الآلة , فسترى مراراً إشارات لمسألة MNIST مجدداً بأوراق علمية منشورات ومدونات وهكذا.
 
 مسألة MNIST تأتي مسبقة التحميل في كيراس ، بصيغة مجموعة من 4 مصفوفات Numpy:
 




In [1]:
from keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

Using TensorFlow backend.


`train_images` `train_labels` تشكل مجموعات التدريب ، وهي البيانات التي سيتعلم منها نموذج الشبكة العصبونية . سيتم اختبار النموذج بعدها على مجموعة الاختبار test set و التي تشمل : `test_images` `test_labels` . الصور ممثلة كمصفوفات Numpy والتأشير ببساطة هو مصفوفة خانات تتراوح من 0 ل 9 . حيث لكل صورة بالمجموعتين هناك رقم مقابل وحيد ضمن مصفوفة التأشيرات labels .

فللنظر لبيانات التدريب:

In [2]:
train_images.shape

(60000, 28, 28)

In [3]:
len(train_labels)

60000

In [4]:
train_labels

array([5, 0, 4, ..., 5, 6, 8], dtype=uint8)

أما بيانات الاختبار:

In [5]:
test_images.shape

(10000, 28, 28)

In [6]:
len(test_labels)

10000

In [7]:
test_labels

array([7, 2, 1, ..., 4, 5, 6], dtype=uint8)

عملنا سيكون كما يلي: أولاً سنعرض شبكتنا العصبونية لبيانات التدريب ، `train_images` , `train_labels` . ستتعلم حينها الشبكة الربط بين الصور والتأشيرات المقابلة وأخيراً سنطلب من الشبكة أن تنتج تنبؤات للتأشيرات من أجل صور الأختبار `test_images` وسنتحقق من أن هذه التنبؤات تطابق التأشيرات الفعلية من `test_labels`.

فلنبني الشبكة , ومجدداً ، ليس من المفترض بك فهم كل تلك التعليمات الآن.

In [8]:
from keras import models
from keras import layers

network = models.Sequential()
network.add(layers.Dense(512, activation='relu', input_shape=(28 * 28,)))
network.add(layers.Dense(10, activation='softmax'))

الحجرة الأساسية لبناء شبكة عصبونية هي الطبقة `layer` وهو نموذج معالجة بيانات يمكنك جعله كمرشح للبيانات. بعض البيانات تدخل ويخرج بصيغة أكثر فائدة . وتحديداً تقوم الطبقة باشتقاق تمثيلات من البيانات المزودة بها  ربما قد تكون أكثر فائدة للمسألة المعطاة . معظم التعلم العميق يتألف عملياً من وصل سلسلة طبقات بسيطة ستطبق صيغة من تحويلات تمثيلات البيانات. نموذج التعلم العميق مماثل لشبكة معالجة بيانات تتألف من تسلسل مرشحات بيانات محسنة باطراد  " الطبقات".

هنا تتألف الشبكة من تسلسل طبقتي Dense والتي ترتبط بشكل كثيف (أوكامل) كعصبونات. الطبقة الثانية ولأخيرة هي طبقة بعشرة مسالك من نوع "softmax" ,مما يعني أنها ستعطي مصفوفة من عشرة قيم احتمالية (مجموعها 1) . كل قيمة ستمثل احتمالية أن تكون صورة الخانة المدخلة عائدة لصنف الخانة المرتبط بتلك القيمة .

لجعل شبكتنا جاهزة للتدريب علينا الانتباه لثلاثة مواضيع تشكل جزءاً من خطوة الترجمة:

* تابع خسارة: وهي طريقة تمكن الشبكة من تحديد مدى جودة قيامها بمهمتها وبالتالي تحديد كيفية قيادة نفسها بالاتجاه الصحيح .

* محسن أمثل: وهذه هي آلية تقوم الشبكة باتباعها لتحديث نفسها بالاعتماد على البيانات التي تراها و تابع الخسارة.

* المعايير التي سنختبرها عبر التدريب والاختبار . هنا سنهتم فقط بالدقة (نسبة الصور التي تم تصنيفها بنجاح).

الهدف الأساسي لتابع الخسارة وللمحسن الأمثل سيظهر بوضوح من خلال الأمثلة اللاحقة.

In [9]:
network.compile(optimizer='rmsprop',
                loss='categorical_crossentropy',
                metrics=['accuracy'])

قبل التدريب ، سنقوم بمعالجة بيانات الدخل لصياغتها بالشكل الذي تتوقعة الشبكة كدخل و وتحديد قيمها ضمن المجال [1,0] .

مسبقاً كانت صور التدريب مخزنة بمصفوفة من الشكل: `(60000,28,28)` ومن النوع `uint8` مع قيم ضمن `[0,255]` . سننقل هذا لمصفوفة من نمط `float32` وبأبعاد `(60000,28*28)` مع قيم بين ال 0 وال 1.

In [10]:
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype('float32') / 255

test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype('float32') / 255

علينا أيضاً جعل التأشيرات كفئات وهي خطوة  تقوم بتديل التأشير كرقم مفرد مثلاً 5 , بمصفوفة من 10 عناصر , كلها أصفار ما عدا العنصر الخامس 1 ، للتوافق مع التمثيل الاحتمالي السابق الإشارة إليه.



In [11]:
from keras.utils import to_categorical

train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

والآن أصبحنا جاهزين لتدريب الشبكة ، والتي سيتم القيام بها في كيراس عبر طريقة `fit` للشبكة , حيث ستلائم النموذج لبيانات تدريبه:

In [12]:
network.fit(train_images, train_labels, epochs=5, batch_size=128)

Epoch 1/5
60000/60000 [==============================] - 13s 211us/step - loss: 0.2568 - acc: 0.92481s - loss: 0.2
Epoch 2/5
60000/60000 [==============================] - 10s 166us/step - loss: 0.1048 - acc: 0.9691
Epoch 3/5
60000/60000 [==============================] - 10s 168us/step - loss: 0.0683 - acc: 0.9790
Epoch 4/5
60000/60000 [==============================] - 10s 171us/step - loss: 0.0500 - acc: 0.9848
Epoch 5/5
60000/60000 [==============================] - 10s 168us/step - loss: 0.0378 - acc: 0.98871s - loss: 0.0380 - acc: 0.9 - ETA: 1s - los


كميتان يتم عرضهما خلال التدريب : الخسارة للشبكة عبر بيانات التدريب ودقة الشبكة عبر بيانات التدريب.

وسريعاً نصل لدقة 0.988 على بيانات التدريب ، والآن ينبغي اختبار النموذج على بيانات الاختبار أيضاَ

In [13]:
test_loss, test_acc = network.evaluate(test_images, test_labels)

10000/10000 [==============================] - 1s 102us/step


In [14]:
print('test_acc:', test_acc)

test_acc: 0.9806


بيانات الاختبار يبدو أنها تعطي دقة 98% وهذا أقل بقليل من دقة بيانات الاختبار . وهذه الفجوة بين دقة التدريب ودقة الاختبار هي مثال على "الملائمة الزائدة عن الحد" وهي الحقيقة المسببة لكون نماذج تعليم الآلة تعطي أداءاَ أسوء على البيانات الجديدة من البيانات القديمة. 

وهذا يختتم مثالنا الأول - ولقد رأيت لتوك كيف يمكن بناء وتدريب شبكة عصبونية لتصنيف البيانات المكتوبة بخط اليد بأقل من 20 سطراً من أكواد بايثون. 
ولاحقاً سنرى بنى اخرى للشبكات العصبونية ونستعرض أمثلة أكثر للأيضاح.